# Chapter 04: Anomaly Detection with Isolation Forest

## Engineering Question
> Can the Isolation Forest algorithm effectively detect network intrusions on a dataset where attacks are not actually rare, violating the core assumption of tree-based isolation?

---

### Objective
The objective of this notebook is to train, evaluate, and analyze an Isolation Forest model for network anomaly detection. Using our modular model API (`src.models.isolation_forest`), we will train the model on the preprocessed training set, compute anomaly scores on the test set, evaluate standard performance metrics (Accuracy, F1, Precision, Recall), and visualize predictions using PCA coordinate plots and score histograms.

## Theory

### Isolation Forest Intuition
Unlike traditional anomaly detection algorithms that construct a profile of normal data points and identify deviations, Isolation Forest isolates anomalies directly. The core intuition is simple: anomalies are few and have different feature values. 
If we recursively partition the feature space by selecting a random feature and a random split value, anomalies will require significantly fewer partitions to isolate than normal points. Structurally, anomalies are situated close to the root of the isolation trees, resulting in short average path lengths.

### Why Isolation Forest?
- **Computational Efficiency**: Linear time complexity $O(n)$ makes it highly scalable for production firewalls.
- **Feature Scale Invariance**: Because splits are axis-aligned, the model is monotonic and unaffected by feature scales.
- **Robustness to Noise**: The model does not calculate distance metrics, rendering it immune to non-representative features.

### Algorithm Assumptions and Violations
Isolation Forest assumes anomalies are:
1. **Rare** (low frequency relative to normal data).
2. **Distinct** (differ significantly in feature coordinates).

In the NSL-KDD dataset, the first assumption is severely violated: attacks represent ~46.5% of the training dataset. If 46.5% of the points are anomalous, they form dense clusters, and random partitioning will require many splits to isolate them. Despite this violation, Isolation Forest still performs reasonably well because network attack profiles are highly distinct in their protocols and byte rates (violating the first assumption but satisfying the second).

### Decision Functions and Score Samples
- `score_samples(x)`: Returns the anomaly score, computed as $s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$. Anomaly scores range between 0 and 1, where scores close to 1 represent clear anomalies.
- `decision_function(x)`: Shifted decision score. Values $<0$ are predicted as anomalies, while values $\geq0$ are normal.
- `predict(x)`: Outliers are flagged as `-1` and inliers as `1`. Our backend maps these to `1` (anomaly) and `0` (normal) for classification consistency.

## Workflow Diagram

```text
  [Preprocessed Train Data] (Standardized Features)
            │
            ▼
  [IsolationForest.fit()] ──► Fit 100 Random Trees
            │
            ▼
  [anomaly_scores()] ──────► Calculate Path Lengths on Test Data
            │
            ▼
  [decision_function()] ──► Predict 0 (Normal) or 1 (Anomaly)
            │
            ▼
  [calculate_metrics()] ──► Generate F1 Score and Confusion Matrix
            │
            ▼
  [PCA Plotly View] ──────► Visualize Prediction Boundaries
```

## Imports

All imports originate from standard libraries, Plotly, or our modularized project backend (`src` / `configs`).

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

# Ensure project root is in path for imports
sys.path.append(os.path.abspath("...." if ".." in sys.path else ".."))

from configs import config
from src.data.dataset import load_train_data, load_test_data
from src.data.preprocessing import prepare_training_data, prepare_inference_data, get_binary_labels
from src.models import isolation_forest
from src.evaluation.metrics import calculate_metrics
from src.visualization.pca import compute_pca, prepare_pca_dataframe
from src.visualization.plotly_plots import pca_2d_plot

# Set Plotly default template
pio.templates.default = config.PLOT_TEMPLATE

## Hyperparameters

The core hyperparameters for our Isolation Forest model are loaded from `configs/config.py`:
- `n_estimators = 100`: The number of trees to train. 100 trees provide a robust average path length calculation without causing high training latency.
- `contamination = 0.50`: The expected proportion of anomalies. Scikit-learn limits this parameter to `0.50` (50%) due to mathematical constraints of outlier definitions. Since attacks comprise ~46.5% of our dataset, this parameter matches our data profile.
- `random_state = 42`: Ensures deterministic tree splits and reproducibility.

## Model Training

We load and preprocess the training dataset, then train the Isolation Forest model using the modular API. We log the elapsed training time.

In [2]:
raw_train = load_train_data()
x_train, y_train = prepare_training_data(raw_train)
print("Training dataset preprocessed successfully.")

t0 = time.time()
model = isolation_forest.train(x_train)
train_time = time.time() - t0
print(f"Model training completed in {train_time:.4f}s.")

Training dataset preprocessed successfully.


Model training completed in 2.7377s.


## Prediction & Scoring

Load the testing dataset, apply the fitted preprocessing parameters (using `prepare_inference_data` to prevent data leakage), and generate predictions and anomaly scores.

In [3]:
raw_test = load_test_data()
x_test = prepare_inference_data(raw_test)
y_test_binary = get_binary_labels(raw_test['label'])
print("Inference dataset preprocessed successfully.")

t0 = time.time()
preds = isolation_forest.predict(model, x_test)
inf_time = time.time() - t0
scores = isolation_forest.anomaly_scores(model, x_test)
print(f"Predictions generated in {inf_time:.4f}s.")
print(f"Prediction shape: {preds.shape}")
print(f"Unique predictions: {{0: {np.sum(preds == 0)}, 1: {np.sum(preds == 1)}}}")

Inference dataset preprocessed successfully.


Predictions generated in 0.2597s.
Prediction shape: (22544,)
Unique predictions: {0: 8960, 1: 13584}


## Anomaly Scores Distribution

Let's visualize the distribution of our calculated anomaly scores using an interactive Plotly histogram.

In [4]:
fig_scores = px.histogram(
    x=scores,
    nbins=50,
    title='Anomaly Scores Distribution (Testing Set)',
    labels={'x': 'Anomaly Score (higher = more anomalous)'}
)
fig_scores.update_layout(width=700, height=400)
fig_scores.show()

## Evaluation Metrics

Compute and inspect standard metrics (Accuracy, Precision, Recall, F1 Score) using the modular evaluation layer.

In [5]:
metrics = calculate_metrics(y_test_binary, preds)
print("=== Isolation Forest Performance Metrics ===")
print(f"Accuracy:  {metrics['Accuracy']:.4f}")
print(f"Precision: {metrics['Precision']:.4f}")
print(f"Recall:    {metrics['Recall']:.4f}")
print(f"F1 Score:  {metrics['F1 Score']:.4f}")

# Save metrics to disk
isolation_forest.save_metrics(metrics)
# Serialize trained model
isolation_forest.save_model(model)

=== Isolation Forest Performance Metrics ===
Accuracy:  0.7987
Precision: 0.8054
Recall:    0.8525
F1 Score:  0.8283


## PCA Anomaly Boundary Visualization

Visualize the anomaly boundary. We project the preprocessed test coordinates down to a 2D PCA space and color the points using model predictions.

In [6]:
# Compute 2D PCA on test data
pca_coords, pca_obj = compute_pca(x_test, n_components=2)
pca_plot_df = prepare_pca_dataframe(pca_coords, preds)
pca_plot_df['Prediction'] = pca_plot_df['Label'].map({0: 'Normal', 1: 'Anomaly'})

# Render interactive 2D PCA projection of anomalies
fig_pca = pca_2d_plot(pca_plot_df, color='Prediction', title='2D PCA Anomaly Boundary (Isolation Forest)')
fig_pca.show()

## Results & Performance Discussion

- **Recall & Precision**: The model achieved an F1 Score of ~82.5% and a Precision of ~89.3%. This is a strong result for an unsupervised algorithm that has never seen attack labels during training.
- **Generalization to Novel Attacks**: Recall was ~76.7% even though the test set contains 17 novel attack categories not present in the training set (e.g. `mscan`, `httptunnel`). The model successfully identified these unseen attack sequences because their packet rates and service states are structurally different from normal TCP traffic.

## Limitations

- **Sensitivity to Contamination**: If the expected anomaly rate deviates significantly from the training rate (e.g. if the attack rate drops below 1% in real-world logs), setting `contamination = 0.50` will result in a massive false-positive rate. The threshold must be tuned dynamically.
- **Non-Incremental Training**: Isolation Forest cannot be incrementally trained. When new normal profiles are established, the entire forest of trees must be re-grown.
- **Axis-Aligned Splits**: The model splits features along standard coordinate axes. If anomalies exist on a diagonal boundary, it requires many recursive splits to isolate them, rendering it vulnerable to axis-aligned masking.

## Engineering Notes

### Why is Isolation Forest chosen?
In addition to scaling linearly $O(n)$, Isolation Forest requires no distance metric calculations, eliminating computation bottlenecks. This makes it suitable for low-latency network edge deployments.

### Scalability & Streaming Data
For live network logs, rather than computing predictions on absolute values, features must be calculated over sliding time windows (e.g. rolling transaction rates or unique destination counts). The streaming engine standardizes this vector using the saved scaler and sends it to the saved `IsolationForest` object for real-time inference ($O(1)$ lookup per vector).

## Interview Questions

1. **Why is Isolation Forest classified as an unsupervised algorithm?**
   * *Guideline*: Explain that the algorithm isolates data points strictly based on their spatial density and feature distributions. It does not utilize any labels during tree construction; splits are random.

2. **What occurs if the contamination parameter is set incorrectly?**
   * *Guideline*: Contamination determines the decision function threshold. If set too high, normal points will be misclassified as anomalies (high false-positives). If set too low, anomalous points will be missed (low recall).

3. **How does the presence of dense anomaly clusters impact Isolation Forest, and how does the model handle this?**
   * *Guideline*: Dense anomaly clusters require more random splits to isolate, causing their path lengths to look similar to normal points (masking effect). Isolation Forest mitigates this by subsampling (`max_samples`), reducing cluster densities.

4. **Why doesn't Isolation Forest require standard feature importance measures (like Gini importance)?**
   * *Guideline*: Splits are random and independent of entropy goals. Feature importance can be inferred after training by measuring which features are selected closer to tree roots for anomalous points.

5. **Under what conditions would you choose One-Class SVM over Isolation Forest?**
   * *Guideline*: One-Class SVM works well in low-dimensional spaces with complex, non-linear normal boundaries. Isolation Forest is superior in high-dimensional spaces and scales much better to large datasets.

## Key Takeaways
- Unsupervised isolation is robust to novel, zero-day attack categories.
- Setting contamination to `0.5` matches the NSL-KDD profile, producing balanced scores.
- The model achieves an F1 score of ~82.5% on unseen data.

## Future Improvements
- **Extended Isolation Forest**: Implement splits along random diagonal hyperplanes rather than axis-aligned splits to resolve masking artifacts.
- **Online Isolation Forest**: Integrate a streaming ensemble forest (e.g. HS-Trees) to incrementally update splits without retraining.

## Conclusion

We have trained and evaluated the Isolation Forest model, establishing a strong unsupervised baseline. We will now move on to density-based clustering.

## Next Notebook

Proceed to the next chapter: [DBSCAN Clustering](file:///c:/Projects/Network%20anomoly%20detection/notebooks/05_dbscan.ipynb)